# 6.7 一個 token 一定是一個完整字嗎？

# 第 6 章：自己的中英文 tokenizer

前置：字元ID與下一字loss。Tokenizer訓練是統計切詞規則，不是訓練LLM。早期與CLI使用可還原ByteTokenizer；BPE在本章獨立學習，換模型時要同步詞表與checkpoint。




In [ ]:
from pathlib import Path
import sys

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("先依 course/README.md 下載並開啟這個 repo")
sys.path.insert(0, str(root))
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)
torch.manual_seed(42)

**先看圖：常見卡片合併：不是先懂詞義**

每次合併一對常見相鄰符號。

橘色邊框只提示閱讀順序，靜態標註一直保留。系統的減少動態效果設定會停用動畫。


In [ ]:
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/bpe.svg")))

**先想一想：**只decode中文字的第一個byte，會得到完整字嗎？

一個Unicode字可由多個UTF-8 byte組成。切到中間byte單獨decode會不完整，串接完整序列再decode纔是正確契約。

**把直覺寫成數學：**「貓」通常3 bytes，emoji通常4；token不是必然等於Unicode字元。

**最小實驗：**以下程式只處理這節的問題。


In [ ]:
raw = "貓🙂".encode()  # 預設就是 UTF-8
print(list(raw), raw[:1].decode("utf-8", errors="replace"))
print(raw.decode("utf-8"))
assert raw.decode() == "貓🙂"

**如何讀結果：**replacement字符不等於資料真的被損毀；可能只是觀察單位太小。

**只改一件事：**只把decode邊界切在完整中文字之後。
